# 公开数据三层验证

**趋势部分相符，房间绝对校准没有成立。** 产品源码冻结。
这是已执行的结果伴随Notebook，读取5090真实结果；完整原始计算在`run_validation.py`，不是再次生成模型结果。


In [1]:
from pathlib import Path
import json,pandas as pd
ROOT=Path.cwd()
while not (ROOT/"operation_planning").is_dir():
    if ROOT==ROOT.parent:raise RuntimeError("Run inside repository")
    ROOT=ROOT.parent
OUT=ROOT/"operation_planning/results/round24_public_validation"
read=lambda name:json.loads((OUT/name).read_text(encoding="utf-8"))
run=read("run_manifest_all.json")
print("Actual source_commit:",run["source_commit"])
print("Actual GPU query:",run["gpu_output"])
print("Model computation is CPU; no LLM called.")


Actual source_commit: 8e1d664758025c971185e21872e39d995c2e610b
Actual GPU query: NVIDIA GeForce RTX 5090, 610.88
Model computation is CPU; no LLM called.


## 方法与关键假设

房间：3个单空调房，19天。9个开发日只选择一次共享U/热容；10个后段评价日。电流是家庭相电流，代理量230V/PF0.9，不是真实空调分表。楼宇只比198个共同完整工作日的归一化形状；城市只比3栋全年电表完整、有冬季气表的行政建筑。全部缺测不补0。详细来源、假设、失败与准入修订在README/protocol.json。


In [2]:
r=read("room_summary.json")
rows=[]
for room in r["rooms"]:
    for variant in ["untuned_evaluation","limited_fit_evaluation"]:
        m=room[variant]
        rows.append({"house":room["house_id"],"variant":variant,"days":m["n_days"],"proxy_NMBE_pct":m["nmbe_percent_model_minus_proxy"],"proxy_CVRMSE_pct":m["cvrmse_percent"],"room_T_RMSE_C":m["temperature_rmse_c"]})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Individually complete room physical parameter sets:",r["rooms_with_individually_complete_physical_parameters"])


 house                variant  days  proxy_NMBE_pct  proxy_CVRMSE_pct  room_T_RMSE_C
     2     untuned_evaluation    10         30.8180           40.3982         3.8008
     2 limited_fit_evaluation    10         54.2353           58.2171         3.3027
     6     untuned_evaluation    10         37.3760           39.2975         5.9440
     6 limited_fit_evaluation    10         60.4102           59.9817         5.0851
     9     untuned_evaluation    10        -15.2617           23.8615         2.9015
     9 limited_fit_evaluation    10          1.7080           19.6953         2.2638
Individually complete room physical parameter sets: 0


![房间：电量代理与室温](../../operation_planning/results/round24_public_validation/figures/room_validation.png)

有限调参室温降低，但两户电量代理误差变大；不宣称总体精度提升。


In [3]:
cu=read("cu_summary.json")
print("Accepted weekdays:",cu["accepted_weekdays"],"of",cu["all_weekdays"])
print("Shape r:",cu["normalized_hourly_shape_correlation"])
print("Measured relative slope per C:",cu["measured_relative_sensitivity"]["relative_slope_per_c"])
print("Model relative slope per C:",cu["model_relative_sensitivity"]["relative_slope_per_c"])


Accepted weekdays: 198 of 261
Shape r: 0.9478538322357675
Measured relative slope per C: 0.056899082203409015
Model relative slope per C: 0.09727154330396977


![楼宇：形状及温度关联](../../operation_planning/results/round24_public_validation/figures/cu_validation.png)

只验证规模无关的趋势；并非人员、季节因素控制后的因果敏感度。


In [4]:
madrid=read("madrid_summary.json")
print("Included/excluded:",madrid["included_buildings"],madrid["excluded_building_names"])
print(pd.read_csv(OUT/"madrid_building_statistics.csv")[["building","signed_increment_cdd18_correlation_12_months","summer_positive_excess_fraction","winter_positive_excess_fraction"]].round(4).to_string(index=False))
print("Gas boundary:",madrid["gas_boundary"])


Included/excluded: 3 25
      building  signed_increment_cdd18_correlation_12_months  summer_positive_excess_fraction  winter_positive_excess_fraction
JMD Arganzuela                                        0.7830                           0.5671                           0.2543
  JMD Chamberí                                        0.7633                           0.5693                           0.1938
 JMD Moratalaz                                        0.9363                           0.7723                           0.0000
Gas boundary: Gas metering does NOT prove predominant gas heating; winter electricity excess and other seasonal loads remain visible


![城市：季节分布](../../operation_planning/results/round24_public_validation/figures/madrid_validation.png)

保留负增量和冬季正超额。不能把正超额或燃气表存在等同于真实空调制冷量。


In [5]:
v=read("verification.json")
print("Analysis contract/reconciliation checks:",v["count"],"passed:",v["all_passed"])
print("These are NOT accuracy or savings validations.")


Analysis contract/reconciliation checks: 23 passed: True
These are NOT accuracy or savings validations.


## 可支持与不可支持的结论

完成公开数据三层对照及冻结模型局限检查；部分日内/季节形状符合，默认温度敏感度偏高，房间室温偏差明显。没有参数完整的房间与独立空调有功分表，不能说已校准、真实节能或已形成现场精度。
